### IRR Example (continue)

In `absbox`, user can calculate irr of a tranche or (multiple tranches) for scenarios of:

* During the waterfall distribution, user want to pay up to x% IRR to a bond (which makes IRR of the bond close to that threshold.
* Using IRR from the formula, like inspect the current IRR of the bond, or use the IRR in a `condition`

#### Related formula

Since engine version : `0.52.3`

* `IrrOfBond`

#### Init

In [1]:
from absbox import examples,API,EnginePath
localAPI = API(EnginePath.DEV, check=False)

Connecting engine server -> http://localhost:8081

✅Connected, local lib:0.51.5, server:0.52.3

In [2]:
from absbox import Generic

Irr02 = Generic(
    "IRR Case 02"
    ,{"cutoff":"2021-03-01","closing":"2021-04-01","firstPay":"2021-06-20"
     ,"payFreq":["DayOfMonth",20],"poolFreq":"MonthFirst","stated":"2030-01-01"}
    ,{'assets':[["Mortgage"
        ,{"originBalance":2200,"originRate":["fix",0.045],"originTerm":20
          ,"freq":"Monthly","type":"Level","originDate":"2021-02-01"}
          ,{"currentBalance":2200
          ,"currentRate":0.08
          ,"remainTerm":20
          ,"status":"current"}]]}
    ,(("acc01",{"balance":0}),)
    ,(("A1",{"balance":1000
             ,"rate":0.07
             ,"originBalance":1000
             ,"originRate":0.07
             ,"startDate":"2021-04-01"
             ,"rateType":{"Fixed":0.08}
             ,"bondType":{"Sequential":None}})
      ,("B",{"balance":1000
             ,"rate":0.0
             ,"originBalance":1000
             ,"originRate":0.07
             ,"startDate":"2021-04-01"
             ,"rateType":{"Fixed":0.00}
             ,"bondType":{"Equity":None}
             ,"stmt":[["2021-04-01",0,0,0,0,-1000,0,0,None,""],]
             }))
    ,(("incentiveFee",{"type":{"fixFee":100},"feeStart":"2021-04-01"}),)
    ,{"amortizing":[
         ["accrueAndPayInt","acc01",["A1"]]
         ,["payPrin","acc01",["A1"]]
         ,["payPrin","acc01",["B"]]
         #,["payIntResidual","acc01","B",{"limit":("ds",("floorWithZero",("amountForTargetIrr",0.11,"B")))}]
         ,["payIntResidual","acc01","B",{"limit":("ds",("floorWithZero",("amountForTargetIrr",0.10,"B")))}]
         ,["payFee","acc01",['incentiveFee',]]
         ,["payFeeResidual","acc01","incentiveFee"]
     ]}
    ,[["CollectedInterest","acc01"]
      ,["CollectedPrincipal","acc01"]
      ,["CollectedPrepayment","acc01"]
      ,["CollectedRecoveries","acc01"]]
    ,None
    ,None
    ,None
    ,None
    ,("PreClosing","Amortizing")
    )


It says , pay out cash from account `acc01` as a residual yield ( which won't be constrainted to interest due ) to the bond `B`, by amount limited to a cap , which is the amount will make irr of bond B equal to `10%`

    ["payIntResidual","acc01","B",{"limit":("ds",("floorWithZero",("amountForTargetIrr",0.10,"B")))}]

* adding a `floorWithZero` , because the `amountForTargetIrr` may return negative


#### Inspect the IRR changes of bond

In [16]:
r0 = localAPI.run(Irr02
                ,poolAssump=("Pool",("Mortgage",{"CDRPadding":[0.01,0.02]},{"CPR":0.02},{"Rate":0.1,"Lag":5},None)
                                ,None
                                ,None)
                ,runAssump = [
                    ("inspect",["MonthEnd",("irrOfBond","B")]
                                )
                ]
                ,read=True
                )

In [18]:
r0['result']['inspect']['<IrrOfBond:B>']

,<IrrOfBond:B>
Date,
2021-03-01,-1.000000
2021-03-31,-1.000000
2021-04-30,-1.000000
2021-05-31,-1.000000
2021-06-30,-1.000000
2021-07-31,-1.000000
2021-08-31,-1.000000
2021-09-30,-1.000000
2021-10-31,-1.000000
